<a href="https://colab.research.google.com/github/Jeremy26/vslam/blob/claude%2Fimprove-vslam-notebook-suMKx/improve-vslam-notebook-suMKx/Visual_SLAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Visual Odometry Workshop**
Welcome to the VO Workshop!

In this workshop, we're going to learn how to use feature tracking to build an odometry estimation algorithm! This is very useful in Visual SLAM systems, often considered Step #1.
<P>
So let's begin! We will do this in 3 steps:
1. Feature Tracking (Detection, Description, Matching)
2. Pose Recovery (E & F, R & T)
3. Visual Odometry Graph

In [ ]:
import pandas as pd, numpy as np

df = pd.read_parquet('testing_vehicle_pose_8993680275027614595_2520_000_2540_000.parquet')
print(df.shape)
print(df.columns.tolist())
print(df.iloc[0])

In [ ]:
ts_col   = 'key.frame_timestamp_micros'
pose_col = '[VehiclePoseComponent].world_from_vehicle.transform'   # confirm from Step 1

df = df.sort_values(ts_col).reset_index(drop=True)

# each cell is a 16-vector; stack -> (N,4,4)
T = np.stack([np.asarray(v, dtype=np.float64).reshape(4, 4)
              for v in df[pose_col].to_numpy()])

ts   = df[ts_col].to_numpy()          # microsecond timestamps
xyz  = T[:, :3, 3]                    # ego position in world (metres!)
gt_xy = xyz[:, :2]                    # top-down ground-truth path

# save a small, library-free artifact you can reuse anywhere
np.savez('gt_pose_8993680275027614595_2520_000_2540_000.npz', ts=ts, T=T, xy=gt_xy)
print(f'{len(ts)} GT poses, span {(ts[-1]-ts[0])/1e6:.1f}s')

In [ ]:
import pyarrow.parquet as pq
import os
import duckdb

PARQUET_PATH = 'testing_camera_image_17136775999940024630_4860_000_4880_000.parquet'
OUTPUT_VIDEO = "testing_camera_image_17136775999940024630_4860_000_4880_000.mp4"
CAM_ID = 1  # FRONT camera
FPS = 10

query = f"""
SELECT
    "key.camera_name",
    "key.frame_timestamp_micros",
    "[CameraImageComponent].image"
FROM '{PARQUET_PATH}'
WHERE "key.camera_name" = {CAM_ID}
ORDER BY "key.frame_timestamp_micros"
"""

df = duckdb.query(query).to_df()

def decode_jpeg(raw):
    return cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)

# Première image
first = decode_jpeg(df.iloc[0]["[CameraImageComponent].image"])
plt.imshow(first)

In [ ]:
df = duckdb.query(query).to_df()

PARQUET_PATH = 'testing_camera_image_17136775999940024630_4860_000_4880_000.parquet'
OUTPUT_VIDEO = "testing_camera_image_17136775999940024630_4860_000_4880_000.mp4"
OUTPUT_DIR = "NEW"
CAM_ID = 1  # FRONT camera
FPS = 10

query = f"""
SELECT
    "key.camera_name",
    "key.frame_timestamp_micros",
    "[CameraImageComponent].image"
FROM '{PARQUET_PATH}'
WHERE "key.camera_name" = {CAM_ID}
ORDER BY "key.frame_timestamp_micros"
"""

def decode_jpeg(raw):
    return cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)

for _, row in df.iterrows():
    ts = int(row["key.frame_timestamp_micros"])
    img = decode_jpeg(row["[CameraImageComponent].image"])

    out_path = os.path.join(OUTPUT_DIR, f"{ts}.jpg")
    cv2.imwrite(out_path, img)

print("Saved images to:", OUTPUT_DIR)

But first, let's do some imports...

## **Waymo Open Dataset & Imports**

In [ ]:
!wget -qq https://optical-flow-data.s3.eu-west-3.amazonaws.com/waymo_images.zip
!unzip -qq waymo_images.zip && rm waymo_images.zip
!mkdir output
!ls

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pickle
from google.colab.patches import cv2_imshow

In [ ]:
#TODO: Load Different Image Pairs
img1 = cv2.imread("downtown/front_images_downtown/1557197711848851.jpg")
img2 = cv2.imread("downtown/front_images_downtown/1557197711948687.jpg")

def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(to_rgb(img2))

The sample images will be used to experiment with the feature tracking, but frankly, in Visual SLAM, we need a complete sequence.

In [ ]:
import json

video_path = 'downtown/front_camera_downtown.mp4'
cap = cv2.VideoCapture(video_path)

# Define the path to the camera calibration JSON file
calibration_json_path = 'downtown/camera_calibration_downtown.json'

print("Video file path: ", video_path)
print("Calibration JSON path: ", calibration_json_path)

# Open and read the JSON file
with open(calibration_json_path, 'r') as f:
    calibration_data = json.load(f)

# Extract camera intrinsics matrix and distortion coefficients
camera_matrix = np.array(calibration_data['intrinsics'])
dist_coeffs = np.array(calibration_data['distortion'])

print("Camera Intrinsics Matrix (K):")
print(camera_matrix)
print("\nDistortion Coefficients:")
print(dist_coeffs)

## **vSLAM Process**

In [ ]:
# INITIALIZE ORB DETECTOR/DESCRIPTOR
orb = cv2.ORB_create(nfeatures=2000)

FLANN_INDEX_KDTREE = 0
index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
search_params = dict(checks = 50)

flann = cv2.FlannBasedMatcher(index_params, search_params)

print("Images loaded and feature objects initialized.")

In [ ]:
# DETECTOR/DESCRIPTOR IMAGE 1
kp1 = orb.detect(img1, None) #detector
kp1, des1 = orb.compute(img1, kp1) #descriptor
#alternatively: kp1, des1 = orb.detectAndCompute(img1, None)


# DETECTOR
kp2, des2 = orb.detectAndCompute(img2, None)

In [ ]:
img1_kp = cv2.drawKeypoints(img1, kp1, None, color=(0,255,0), flags=0)
img2_kp = cv2.drawKeypoints(img2, kp2, None, color=(0,255,0), flags=0)

fig = plt.figure(figsize=(15, 7))
ax1 = plt.subplot(121)
ax2 = plt.subplot(122)

ax1.imshow(to_rgb(img1_kp))
ax2.imshow(to_rgb(img2_kp))
plt.show()

In [ ]:
# FLANN MATCHING
# The `flann` matcher was already created in the setup cell above, so we just use it.
matches = flann.knnMatch(np.float32(des1), np.float32(des2), k=2)  # NP.FLOAT32 needed for ORB/BRIEF

# Keep only confident matches using Lowe's ratio test
good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good.append(m)

print(f"Found {len(good)} good matches after Lowe's ratio test.")

In [ ]:
draw_params = dict(matchColor=(0, 255, 0),   # draw matches in green
                   singlePointColor=None,
                   flags=2)

img_briefmatch = cv2.drawMatches(img1, kp1, img2, kp2, good, None, **draw_params)
cv2_imshow(img_briefmatch)

In [ ]:
p1 = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
p2 = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)

print(f"Extracted {len(p1)} points from img1 (p1) and {len(p2)} points from img2 (p2).")
print("Camera matrix and distortion coefficients are ready for use.")

In [ ]:
p1_undistorted = cv2.undistortPoints(p1, camera_matrix, dist_coeffs, P=camera_matrix)
p2_undistorted = cv2.undistortPoints(p2, camera_matrix, dist_coeffs, P=camera_matrix)

# Reshape to 2D array for cv2.findEssentialMat
p1_undistorted_flat = p1_undistorted.reshape(-1, 2)
p2_undistorted_flat = p2_undistorted.reshape(-1, 2)

# Estimate Essential Matrix
E, mask_E = cv2.findEssentialMat(p1_undistorted_flat, p2_undistorted_flat, camera_matrix, method=cv2.RANSAC, prob=0.999, threshold=1.0)

# Recover Pose
_, R_rel, t_rel, mask_pose = cv2.recoverPose(E, p1_undistorted_flat, p2_undistorted_flat, camera_matrix, mask=mask_E)

print("Matched points undistorted.")
print("Essential Matrix estimated.")
print("Relative Rotation (R_rel) and Translation (t_rel) recovered.")
print("R_rel:\n", R_rel)
print("t_rel:\n", t_rel)

> **⚠️ Scale ambiguity (important!)**
>
> `cv2.recoverPose` returns a translation `t_rel` that is a **unit vector**
> (`‖t_rel‖ = 1`). A single camera cannot tell the difference between a small
> object up close and a large object far away, so monocular VO recovers the
> **direction** of motion but not its magnitude.
>
> Consequence: the trajectory shape is correct, but it is **not metric** — you
> can't read distances in meters off the plot. Real systems fix this with a
> stereo rig, wheel odometry, IMU, or a known object size. We'll just assume a
> unit step per frame here.

In [ ]:
import matplotlib.pyplot as plt

# Define the initial camera position
initial_camera_pos = (0, 0)

# Extract X and Z components from t_rel
# t_rel is a 3x1 vector: [[X], [Y], [Z]]
# For a top-down view, we are interested in X (lateral) and Z (forward) movement.
# The coordinate system assumes Z is forward, X is right, and Y is down.
# So, t_rel[0] is X, t_rel[2] is Z.
second_camera_x = t_rel[0, 0]
second_camera_z = t_rel[2, 0]

# Create a new Matplotlib figure
plt.figure(figsize=(8, 8))

# Plot the initial camera position
plt.plot(initial_camera_pos[0], initial_camera_pos[1], 'ro', markersize=10, label='Camera 1 Position (Origin)')

# Plot the second camera's position
plt.plot(second_camera_x, second_camera_z, 'go', markersize=10, label='Camera 2 Position')

# Draw a dashed blue line connecting the two positions
plt.plot([initial_camera_pos[0], second_camera_x], [initial_camera_pos[1], second_camera_z], 'b--', label='Relative Translation')

# Set labels and title
plt.xlabel('X-coordinate (Lateral Movement)')
plt.ylabel('Z-coordinate (Forward Movement)')
plt.title('2D Plot of Camera Positions for Two Frames (Top-Down View)')

# Add a grid
plt.grid(True)

# Ensure equal scaling for the X and Z axes
plt.axis('equal')

# Display the legend
plt.legend()

# Show the plot
plt.show()

In [ ]:
trajectory_points = []
trajectory_points.append(t_rel.flatten())

### **Exercise: Add a 3rd Camera?**

In [ ]:
img3 = cv2.imread("downtown/front_images_downtown/1557197712048522.jpg")
# DETECTOR
kp3 = #TODO
# DESCRIPTOR
kp3, des3 = #TODO
img3_kp = #TODO

plt.imshow(to_rgb(img3_kp))
plt.show()

In [ ]:
### MATCH
matches = #TODO

good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        #TODO

print(f"Found {len(good)} good matches after Lowe's ratio test.")
img_briefmatch23 = #TODO
cv2_imshow(img_briefmatch23)

In [ ]:
# The 'good' matches here refer to the matches between des2 and des3 from the previous cell OilWm56R-YYj

# Extract points from kp2 using queryIdx from the good matches (des2 -> des3)
p2_for_img2img3 = #TODO
p3_for_img2img3 = #TODO

# Undistort points for the current match
p2_undistorted_for_img2img3 = #TODO
p3_undistorted_for_img2img3 = #TODO

# Reshape to 2D array for cv2.findEssentialMat
p2_undistorted_flat_for_img2img3 = #TODO
p3_undistorted_flat_for_img2img3 = #TODO

# Estimate Essential Matrix
E, mask_E = #TODO
# Recover Pose
_, R_rel_23, t_rel_23, mask_pose = #TODO

print("Recovered Relative Rotation Matrix (R_rel_23):\n", R_rel_23)
print("Recovered Relative Translation Vector (t_rel_23):\n", t_rel_23)


In [ ]:
# EXERCISE: accumulate the trajectory CORRECTLY.
#
# cv2.recoverPose returns (R_rel, t_rel) with  X2 = R_rel @ X1 + t_rel
# (it maps a point from the PREVIOUS camera frame into the CURRENT one,
#  and t_rel is a UNIT vector -- monocular has no scale).
#
# The CAMERA CENTRE in a common WORLD frame therefore chains as:
#
#     Rwc = Rwc @ R_rel.T          # camera -> world orientation
#     C   = C   - Rwc @ t_rel      # camera centre in world
#
# (Using R_world @ t_rel with the world->cam matrix -- no transpose, wrong
#  sign -- only mirrors under straight motion but bends the path the wrong
#  way as soon as the car turns.)
#
# Reminder: t_rel is up to scale (||t_rel|| == 1), so distances are NOT
# metric here -- only the SHAPE of the path is meaningful.

poses = []

Rwc = np.eye(3)            # camera -> world rotation
C = np.zeros((3, 1))       # camera centre in world
poses.append(C.flatten())

# Camera 1 -> Camera 2
Rwc = #TODO
C   = #TODO
poses.append(C.flatten())

# Camera 2 -> Camera 3
Rwc = #TODO
C   = #TODO
poses.append(C.flatten())

poses = np.array(poses)

plt.figure(figsize=(8, 8))
plt.plot(poses[:, 0], poses[:, 2], 'k--', label='Trajectory Path')
plt.scatter(poses[:, 0], poses[:, 2], c=['r', 'g', 'b'], s=90, zorder=3)
for i, (x, _, z) in enumerate(poses):
    plt.annotate(f'Cam {i + 1}', (x, z), textcoords="offset points", xytext=(8, 8))
plt.xlabel('X (lateral movement)')
plt.ylabel('Z (forward movement)')
plt.title('Accumulated Camera Positions (Top-Down) -- Proper Pose Chaining')
plt.grid(True)
plt.axis('equal')
plt.legend()
plt.show()


---

### **Solution**

In [ ]:
img3 = cv2.imread("downtown/front_images_downtown/1557197712048522.jpg")
# DETECTOR
kp3, des3 = orb.detectAndCompute(img3,None)
img3_kp = cv2.drawKeypoints(img3, kp3, None, color=(0,255,0), flags=0)

plt.imshow(to_rgb(img3_kp))
plt.show()

In [ ]:
### MATCH
matches = flann.knnMatch(np.float32(des2), np.float32(des3), k=2)

good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good.append(m)

print(f"Found {len(good)} good matches after Lowe's ratio test.")
img_briefmatch23 = cv2.drawMatches(img2,kp2,img3,kp3,good,None,**draw_params)
cv2_imshow(img_briefmatch23)

In [ ]:
# The 'good' matches here refer to the matches between des2 and des3 from the previous cell OilWm56R-YYj

# Extract points from kp2 using queryIdx from the good matches (des2 -> des3)
p2_for_img2img3 = np.float32([ kp2[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
p3_for_img2img3 = np.float32([ kp3[m.trainIdx].pt for m in good ]).reshape(-1,1,2)

# Undistort points for the current match
p2_undistorted_for_img2img3 = cv2.undistortPoints(p2_for_img2img3, camera_matrix, dist_coeffs, P=camera_matrix)
p3_undistorted_for_img2img3 = cv2.undistortPoints(p3_for_img2img3, camera_matrix, dist_coeffs, P=camera_matrix)

# Reshape to 2D array for cv2.findEssentialMat
p2_undistorted_flat_for_img2img3 = p2_undistorted_for_img2img3.reshape(-1, 2)
p3_undistorted_flat_for_img2img3 = p3_undistorted_for_img2img3.reshape(-1, 2)

# Estimate Essential Matrix
E, mask_E = cv2.findEssentialMat(p2_undistorted_flat_for_img2img3, p3_undistorted_flat_for_img2img3, camera_matrix, method=cv2.RANSAC, prob=0.999, threshold=1.0)

# Recover Pose
_, R_rel_23, t_rel_23, mask_pose = cv2.recoverPose(E, p2_undistorted_flat_for_img2img3, p3_undistorted_flat_for_img2img3, camera_matrix, mask=mask_E)

print("Recovered Relative Rotation Matrix (R_rel_23):\n", R_rel_23)
print("Recovered Relative Translation Vector (t_rel_23):\n", t_rel_23)


In [ ]:
# SOLUTION: correct camera-centre chaining.
#   Rwc = Rwc @ R_rel.T     (camera -> world orientation)
#   C   = C   - Rwc @ t_rel (camera centre in world; t_rel is unit-scale)
poses = []

Rwc = np.eye(3)
C = np.zeros((3, 1))
poses.append(C.flatten())

# Camera 1 -> Camera 2
Rwc = Rwc @ R_rel.T
C   = C - Rwc @ t_rel
poses.append(C.flatten())

# Camera 2 -> Camera 3
Rwc = Rwc @ R_rel_23.T
C   = C - Rwc @ t_rel_23
poses.append(C.flatten())

poses = np.array(poses)

plt.figure(figsize=(8, 8))
plt.plot(poses[:, 0], poses[:, 2], 'k--', label='Trajectory Path')
plt.scatter(poses[:, 0], poses[:, 2], c=['r', 'g', 'b'], s=90, zorder=3)
for i, (x, _, z) in enumerate(poses):
    plt.annotate(f'Cam {i + 1}', (x, z), textcoords="offset points", xytext=(8, 8))
plt.xlabel('X (lateral movement)')
plt.ylabel('Z (forward movement)')
plt.title('Accumulated Camera Positions (Top-Down) -- Proper Pose Chaining')
plt.grid(True)
plt.axis('equal')
plt.legend()
plt.show()


## **Video Challenge: Full Visual Odometry on a Real Sequence**

Two frames are a warm-up. Real Visual Odometry runs the pipeline over an
**entire video**, chaining every relative pose into one continuous trajectory.

For each consecutive frame pair `(prev → curr)` you will:

1. **Detect + describe** ORB features on the new frame
2. **Match** against the previous frame (FLANN + Lowe's ratio test)
3. **Undistort** the matched points using the camera calibration
4. **Estimate** the Essential matrix (RANSAC) and **recover** `R_rel, t_rel`
5. **Accumulate** the global pose by *chaining* (same math as the 3-camera exercise)

The cell below sets up everything you need. Then complete the exercise loop —
the worked solution and a polished trajectory plot follow.

In [ ]:
import cv2
import numpy as np
import json
import matplotlib.pyplot as plt

# Global camera pose, expressed in the world frame
Rwc = np.eye(3)            # camera -> world rotation, starts at identity
C = np.zeros((3, 1))       # camera centre in world, starts at the origin

# Trajectory storage (list of camera centres in the world frame)
trajectory_points = [C.flatten()]

# Video + calibration for the "city" sequence
video_path = 'city/front_camera_city.mp4'
calibration_json_path = 'city/camera_calibration_city.json'

with open(calibration_json_path, 'r') as f:
    calibration_data = json.load(f)
camera_matrix = np.array(calibration_data['intrinsics'])
dist_coeffs = np.array(calibration_data['distortion'])

# ORB detector/descriptor (same as the two-frame pipeline)
orb = cv2.ORB_create(nfeatures=2000)

# FLANN matcher
FLANN_INDEX_KDTREE = 0
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)

print("Visual Odometry pipeline initialized:")
print(f"  video       : {video_path}")
print(f"  calibration : {calibration_json_path}")
print(f"  detector    : ORB (2000 features) + FLANN matcher")

In [ ]:
# EXERCISE: run Visual Odometry over the whole video.
# Fill in every #TODO. Re-run the setup cell above first so the pose resets.

cap = cv2.VideoCapture(video_path)

ret, prev_frame = cap.read()
assert ret, "Could not read the first frame of the video."

# Features on the very first frame
kp_prev, des_prev = #TODO  detect + describe on prev_frame

frame_idx = 0
while True:
    ret, curr_frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    kp_curr, des_curr = #TODO  detect + describe on curr_frame

    # Skip degenerate frames (nothing to match against)
    if des_prev is None or des_curr is None or len(kp_curr) < 8:
        prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr
        continue

    matches = #TODO  flann.knnMatch(...) with k=2
    good = []
    for pair in matches:
        if len(pair) != 2:          # FLANN can return <2 neighbours
            continue
        m, n = pair
        if m.distance < 0.7 * n.distance:
            good.append(m)

    if len(good) < 8:               # need >=5 for the Essential matrix; keep margin
        prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr
        continue

    p_prev = #TODO  points from kp_prev via m.queryIdx, shape (-1, 1, 2)
    p_curr = #TODO  points from kp_curr via m.trainIdx, shape (-1, 1, 2)

    p_prev_u = #TODO  cv2.undistortPoints(...) then reshape(-1, 2)
    p_curr_u = #TODO

    E, mask_E = #TODO  cv2.findEssentialMat(... RANSAC ...)
    if E is None or E.shape != (3, 3):
        prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr
        continue

    _, R_rel, t_rel, _ = #TODO  cv2.recoverPose(...)

    # Accumulate the camera centre (monocular -> unit scale)
    #TODO  Rwc = Rwc @ R_rel.T ; C = C - Rwc @ t_rel  (see the 3-camera solution)
    trajectory_points.append(C.flatten())

    prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr

cap.release()
trajectory_points = np.array(trajectory_points)
print(f"Processed {frame_idx} frames -> {len(trajectory_points)} poses estimated.")

---
### **Solution**

### **Live Side-by-Side: Camera Feed + Odometry**

Instead of a single static plot, we render a video that mimics what a real
SLAM system shows on screen:

- **Left panel** — the camera feed with the tracked features drawn on it
  (green lines show how each point moved between frames).
- **Right panel** — the odometry map building up in real time as the car drives.

Two robustness tweaks make the trajectory believable on real driving footage:

- **Stationary rejection** — at a red light the car isn't moving, but
  `recoverPose` still returns a bogus unit translation. If the median feature
  motion is tiny we treat the car as *stopped* and hold the pose (you'll see a
  `STOPPED` badge), which removes the tangled scribble at the start.
- **Speed proxy** — instead of a fixed unit step, we scale each step by the
  median pixel motion (a crude speedometer). Still **not metric**, but the path
  geometry is far less distorted than equal-length steps.

Done in **two passes**: first run the VO to learn the full trajectory (so the
map axes are stable), then composite both panels into an MP4 and play it inline.

In [ ]:
import cv2
import numpy as np

# ------------------------------------------------------------------ #
# PASS 1 — run VO over the whole clip.                                 #
#                                                                      #
# Pose accumulation (the part that was wrong before):                  #
#   cv2.recoverPose returns (R_rel, t_rel) with X2 = R_rel·X1 + t_rel  #
#   (maps a point from the PREVIOUS camera frame into the CURRENT one, #
#    t_rel is a UNIT vector — monocular has no scale).                 #
#   The camera CENTRE in the world therefore updates as:               #
#       Rwc = Rwc @ R_rel.T          # camera -> world orientation     #
#       C   = C   - Rwc @ t_rel      # camera centre in world          #
#   The old `t_world += R_world @ t_rel; R_world = R_rel @ R_world`     #
#   used the world->cam matrix (not its transpose) with the wrong      #
#   sign: it only mirrors under pure forward motion but bends the path #
#   the wrong way as soon as the car turns (the "drives straight yet   #
#   curves" artifact).                                                 #
#                                                                      #
# STATIONARY REJECTION: if the median feature motion is tiny the car   #
# is stopped (red light) — recoverPose would still emit a bogus unit   #
# step, so we hold the pose. One position is stored PER FRAME (held    #
# while stopped) so the VO path stays frame-synced with the 10 Hz GT.  #
# A constant unit step is used per moving frame; the true metric scale #
# is recovered later by Umeyama against the ground truth.              #
# ------------------------------------------------------------------ #
STOP_PX = 1.2

Rwc = np.eye(3)            # camera -> world rotation
C = np.zeros((3, 1))       # camera centre in world

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 10.0
ret, prev_frame = cap.read()
assert ret, "Could not read the first frame of the video."
H, W = prev_frame.shape[:2]
kp_prev, des_prev = orb.detectAndCompute(prev_frame, None)

positions = [(float(C[0, 0]), float(C[2, 0]))]   # (x=right, z=forward) per frame
tracks = [None]                                  # (pts_prev, pts_curr, moving) | None
n_moving = n_stopped = 0

while True:
    ret, curr_frame = cap.read()
    if not ret:
        break
    kp_curr, des_curr = orb.detectAndCompute(curr_frame, None)
    track = None

    if des_prev is not None and des_curr is not None and len(kp_curr) >= 8:
        matches = flann.knnMatch(np.float32(des_prev), np.float32(des_curr), k=2)
        good = []
        for pair in matches:
            if len(pair) != 2:
                continue
            m, n = pair
            if m.distance < 0.7 * n.distance:
                good.append(m)

        if len(good) >= 8:
            pts_prev = np.float32([kp_prev[m.queryIdx].pt for m in good])
            pts_curr = np.float32([kp_curr[m.trainIdx].pt for m in good])
            med_disp = float(np.median(np.linalg.norm(pts_curr - pts_prev, axis=1)))
            moving = med_disp >= STOP_PX

            if moving:
                p_prev_u = cv2.undistortPoints(pts_prev.reshape(-1, 1, 2),
                                               camera_matrix, dist_coeffs, P=camera_matrix).reshape(-1, 2)
                p_curr_u = cv2.undistortPoints(pts_curr.reshape(-1, 1, 2),
                                               camera_matrix, dist_coeffs, P=camera_matrix).reshape(-1, 2)
                E, mask_E = cv2.findEssentialMat(p_prev_u, p_curr_u, camera_matrix,
                                                 method=cv2.RANSAC, prob=0.999, threshold=1.0)
                if E is not None and E.shape == (3, 3):
                    _, R_rel, t_rel, _ = cv2.recoverPose(E, p_prev_u, p_curr_u,
                                                         camera_matrix, mask=mask_E)
                    # correct camera-centre chaining (unit step)
                    Rwc = Rwc @ R_rel.T
                    C = C - Rwc @ t_rel
                    n_moving += 1
            else:
                n_stopped += 1
            track = (pts_prev, pts_curr, moving)

    positions.append((float(C[0, 0]), float(C[2, 0])))
    tracks.append(track)
    prev_frame, kp_prev, des_prev = curr_frame, kp_curr, des_curr

cap.release()
positions = np.array(positions)
print(f"Pass 1 done: {len(positions)} frames | {n_moving} moving, {n_stopped} stopped (held).")

# ------------------------------------------------------------------ #
# Align the per-frame VO path to the METRIC ground truth.              #
# Pair vo[k] <-> gt[k] BY FRAME INDEX (both per-frame @ 10 Hz on the   #
# same clip). Umeyama recovers scale+rotation+offset so GT and VO      #
# live in one metric frame and can share a panel.                      #
# ------------------------------------------------------------------ #
clip = '8993680275027614595_2520_000_2540_000_city'   # 'city' | 'downtown' | 'night'
_gt = np.load(f'gt_pose_{clip}.npz')
gt = _gt['xy'].astype(float)
gt_T = _gt['T'].astype(float)
vo = positions[:, :2]
n_vo, n_gt = len(vo), len(gt)
N = min(n_vo, n_gt)
if abs(n_vo - n_gt) > 2:
    print(f"WARNING: VO has {n_vo} frames but GT has {n_gt} poses "
          f"(diff {abs(n_vo - n_gt)}). They are likely NOT the same Waymo "
          f"segment / frame rate -- the overlay is only meaningful when these "
          f"match. Truncating to the first {N} frames anyway.")
vo_p, gt_p = vo[:N], gt[:N]

def umeyama(src, dst):
    ms, md = src.mean(0), dst.mean(0)
    s0, d0 = src - ms, dst - md
    Cm = (d0.T @ s0) / len(src)
    U, D, Vt = np.linalg.svd(Cm)
    S = np.eye(2)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[-1, -1] = -1
    Rm = U @ S @ Vt
    sc = np.trace(np.diag(D) @ S) / ((s0 ** 2).sum() / len(src))
    tt = md - sc * Rm @ ms
    return sc, Rm, tt

sc, Rm, tt = umeyama(vo_p, gt_p)
vo_aligned = (sc * (Rm @ vo_p.T)).T + tt
ate = np.sqrt((np.linalg.norm(vo_aligned - gt_p, axis=1) ** 2).mean())
print(f"metric scale {sc:.3f}  |  ATE {ate:.2f} m  |  paired {N} frames")

# ------------------------------------------------------------------ #
# Canonical Cartesian view: re-zero to the start and rotate so the     #
# vehicle's INITIAL heading points UP. Pure rotation (det +1) => no    #
# mirroring, so a real left turn stays a left turn.                    #
# ------------------------------------------------------------------ #
fwd = gt_T[0, :3, 0]                                  # vehicle +x (forward) in world
theta = -np.arctan2(fwd[1], fwd[0]) + np.pi / 2
R2 = np.array([[np.cos(theta), -np.sin(theta)],
               [np.sin(theta),  np.cos(theta)]])
origin = gt_p[0].copy()
gt_c = (gt_p - origin) @ R2.T
vo_c = (vo_aligned - origin) @ R2.T

# ------------------------------------------------------------------ #
# EQUAL-ASPECT world -> pixel mapping (square units, +x right, +y up). #
# ------------------------------------------------------------------ #
allxy = np.vstack([gt_c, vo_c])
xmin, xmax = allxy[:, 0].min(), allxy[:, 0].max()
ymin, ymax = allxy[:, 1].min(), allxy[:, 1].max()
M = 40
pad = 1.06                                  # smaller => more zoomed-in
xr = max(xmax - xmin, 1.0) * pad
yr = max(ymax - ymin, 1.0) * pad
cx, cy = (xmin + xmax) / 2.0, (ymin + ymax) / 2.0
# equal aspect: a single scale, fit to whichever axis is tighter
scale = min((W - 2 * M) / xr, (H - 2 * M) / yr)

def world_to_px(x, y):
    u = int(W / 2 + (x - cx) * scale)
    v = int(H / 2 - (y - cy) * scale)   # +y up => Cartesian
    return u, v

gt_px = [world_to_px(x, y) for x, y in gt_c]
vo_px = [world_to_px(x, y) for x, y in vo_c]

# ------------------------------------------------------------------ #
# Dark-mode helpers (BGR). Theme: deep navy bg + neon cyan/green.      #
# ------------------------------------------------------------------ #
BG        = (28, 22, 18)
GRID      = (52, 42, 34)
CYAN      = (255, 209, 64)
CYAN_DIM  = (120, 96, 28)
GREEN     = (120, 255, 80)
GREEN_DIM = (40, 110, 30)
RED       = (90, 90, 255)
AMBER     = (60, 180, 255)
TEXT      = (235, 235, 235)
PILL      = (15, 12, 10)

def hud_text(img, text, org, scale_=0.7, color=TEXT):
    """Text on an OPAQUE dark pill (covers the video's burned-in timestamp)."""
    (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale_, 2)
    x, y = org
    pad = 8
    cv2.rectangle(img, (x - pad, y - th - pad), (x + tw + pad, y + base + pad), PILL, -1)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale_, color, 2, cv2.LINE_AA)

def glow_polyline(img, pts, color, dim):
    if len(pts) < 2:
        return
    arr = np.array(pts, np.int32)
    cv2.polylines(img, [arr], False, dim, 7, cv2.LINE_AA)
    cv2.polylines(img, [arr], False, color, 2, cv2.LINE_AA)

def glow_dot(img, center, color, r=6):
    cv2.circle(img, center, r + 4, tuple(int(c * 0.45) for c in color), -1, cv2.LINE_AA)
    cv2.circle(img, center, r, color, -1, cv2.LINE_AA)

# Pre-build the dark backdrop: subtle grid + faint full GT route for context.
odo_bg = np.full((H, W, 3), BG, np.uint8)
for gx in range(M, W - M + 1, max(1, (W - 2 * M) // 8)):
    cv2.line(odo_bg, (gx, M), (gx, H - M), GRID, 1, cv2.LINE_AA)
for gy in range(M, H - M + 1, max(1, (H - 2 * M) // 8)):
    cv2.line(odo_bg, (M, gy), (W - M, gy), GRID, 1, cv2.LINE_AA)
cv2.rectangle(odo_bg, (M, M), (W - M, H - M), GRID, 1, cv2.LINE_AA)

# ------------------------------------------------------------------ #
# PASS 2 — re-read the video (decode only) and composite.             #
#   LEFT  : camera feed + tracked features                            #
#   RIGHT : VO position (cyan) + ground truth (green), metric-aligned  #
# ------------------------------------------------------------------ #
out_path = 'output/vo_sidebyside.mp4'
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W * 2, H))

cap = cv2.VideoCapture(video_path)
n_frames = len(tracks)
idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    k = min(idx, N - 1)   # paired GT/VO index for this video frame

    # --- LEFT: camera, darkened, with neon feature tracks ---
    cam = (frame * 0.78).astype(np.uint8)
    tr = tracks[idx] if idx < len(tracks) else None
    moving = True
    if tr is not None:
        pts_prev, pts_curr, moving = tr
        for (x0, y0), (x1, y1) in zip(pts_prev[:200], pts_curr[:200]):
            cv2.line(cam, (int(x0), int(y0)), (int(x1), int(y1)), GREEN, 1, cv2.LINE_AA)
            cv2.circle(cam, (int(x1), int(y1)), 2, GREEN, -1, cv2.LINE_AA)
    n_feat = 0 if tr is None else len(tr[0])
    hud_text(cam, "CAMERA + FEATURES", (15, 34), 0.7)
    hud_text(cam, f"frame {idx:04d}  |  {n_feat} tracks", (15, 72), 0.55)

    # --- RIGHT: metric Cartesian map, GT vs VO building up together ---
    odo = odo_bg.copy()
    glow_polyline(odo, gt_px[:k + 1], GREEN, GREEN_DIM)
    glow_polyline(odo, vo_px[:k + 1], CYAN, CYAN_DIM)
    glow_dot(odo, gt_px[0], GREEN, 6)
    glow_dot(odo, gt_px[k], GREEN, 6)
    glow_dot(odo, vo_px[k], RED, 7)
    hud_text(odo, "POSITION  -  VO vs GROUND TRUTH", (15, 34), 0.7, CYAN)
    hud_text(odo, f"green = GT (metric)   cyan = VO   ATE {ate:.1f} m", (15, 72), 0.5)
    if not moving:
        hud_text(odo, "[ STOPPED - pose held ]", (15, 110), 0.55, AMBER)

    # progress bar along the bottom
    bx0, bx1, by = M, W - M, H - 6
    cv2.line(odo, (bx0, by), (bx1, by), GRID, 3, cv2.LINE_AA)
    prog = int(bx0 + (bx1 - bx0) * (idx / max(1, n_frames - 1)))
    cv2.line(odo, (bx0, by), (prog, by), CYAN, 3, cv2.LINE_AA)

    canvas = np.hstack([cam, odo])
    cv2.line(canvas, (W, 0), (W, H), (10, 10, 10), 2)  # divider
    writer.write(canvas)
    idx += 1

cap.release()
writer.release()
print(f"Pass 2 done: wrote {idx} frames -> {out_path}")


In [ ]:
# Play the side-by-side result inline (re-encode to H.264 so the browser
# can play it; fall back to the raw mp4v file if ffmpeg isn't available).
import os
from base64 import b64encode
from IPython.display import HTML, display

src = 'output/vo_sidebyside.mp4'
play = 'output/vo_sidebyside_h264.mp4'
if os.system(f'ffmpeg -y -loglevel error -i {src} -vcodec libx264 -pix_fmt yuv420p {play}') != 0:
    play = src

b64 = b64encode(open(play, 'rb').read()).decode()
display(HTML(f'''
<div style="background:#0c0a09;padding:16px;border-radius:12px;
            box-shadow:0 0 24px #000;display:inline-block">
  <video width="900" controls
         style="border-radius:8px;display:block">
    <source src="data:video/mp4;base64,{b64}" type="video/mp4">
  </video>
</div>'''))